# CLIP K-Fold CV — Config Selection

Selects best unfreezing config via 5-fold stratified CV.
See `status.md` for full plan.

In [ ]:
import os, glob, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import open_clip
from sklearn.model_selection import StratifiedKFold

print('Imports OK')

In [ ]:
# ── Config ──────────────────────────────────────────────────
SEED             = 42
K_FOLDS          = 5
NUM_CLASSES      = 100
BATCH_SIZE       = 64
NUM_WORKERS      = 0
PATIENCE_A       = 10   # early stopping patience — Phase A
PATIENCE_B       = 5    # early stopping patience — Phase B
MAX_EPOCHS_A     = 60   # ceiling; actual budget set by CV median
MAX_EPOCHS_B     = 30   # ceiling; actual budget set by CV median
LR_HEAD          = 1e-3
WEIGHT_DECAY     = 1e-4
USE_AUG          = False  # toggle: horizontal flip only (CLIP-safe); does NOT affect head_only

CLIP_MODEL      = 'ViT-B-32'
CLIP_PRETRAINED = 'openai'

DATA_DIR  = './data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
CKPT_DIR  = './checkpoints_kfold'
os.makedirs(CKPT_DIR, exist_ok=True)

# lr_backbone included per config so CV compares LRs directly
# 1-SE rule uses std/sqrt(K) — the strict textbook variant
CONFIGS = [
    {'name': 'head_only',      'unfreeze_blocks': [],           'lr_backbone': 1e-6},
    {'name': 'unfreeze_2_1e6', 'unfreeze_blocks': [11, 10],     'lr_backbone': 1e-6},
    {'name': 'unfreeze_2_1e5', 'unfreeze_blocks': [11, 10],     'lr_backbone': 1e-5},
    {'name': 'unfreeze_4_1e5', 'unfreeze_blocks': [11,10,9,8],  'lr_backbone': 1e-5},
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print('device:', device)

In [ ]:
# ── Load CLIP + save original weights ───────────────────────
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
clip_model = clip_model.to(device)
clip_model.eval()

# Save original backbone state — restored before each fold's Phase B
original_clip_state = {k: v.clone().cpu() for k, v in clip_model.state_dict().items()}

print(f'Model: {CLIP_MODEL} | params: {sum(p.numel() for p in clip_model.parameters()):,}')

In [ ]:
# ── Dataset classes ─────────────────────────────────────────

class LabeledDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label


class EmbeddingDataset(Dataset):
    """Pre-computed CLIP embeddings — Phase A head training."""
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels     = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class SimplePathDataset(Dataset):
    """Used only for embedding extraction."""
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


class TestDataset(Dataset):
    def __init__(self, test_dir, transform):
        self.paths = sorted(
            glob.glob(os.path.join(test_dir, '*.jpg')),
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
        )
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), os.path.basename(self.paths[idx])


print('Dataset classes defined.')

In [ ]:
# ── Build full sample list ───────────────────────────────────
all_paths, all_labels = [], []
for class_id in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_id))
    for fname in sorted(os.listdir(class_dir)):
        if fname.endswith('.jpg'):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f'Total training images: {len(all_paths)} across {NUM_CLASSES} classes')

In [ ]:
# ── Extract cached embeddings (run once) ────────────────────
@torch.no_grad()
def extract_all_embeddings(paths, batch_size=64):
    clip_model.eval()
    ds     = SimplePathDataset(paths, transform=preprocess)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    embs   = []
    for imgs in tqdm(loader, desc='Extracting embeddings'):
        imgs = imgs.to(device)
        embs.append(clip_model.encode_image(imgs).float().cpu())
    return torch.cat(embs, dim=0)

all_embeddings = extract_all_embeddings(all_paths)
print(f'Embeddings shape: {all_embeddings.shape}')  # (N, 512)

In [ ]:
# ── Loss, head, classifier, helper functions ─────────────────

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


class EarlyStopping:
    def __init__(self, patience):
        self.patience   = patience
        self.best_val   = -1.0
        self.counter    = 0
        self.best_epoch = 0
        self.best_state = None
        self._epoch     = 0

    def step(self, val_acc, model):
        self._epoch += 1
        if val_acc > self.best_val:
            self.best_val   = val_acc
            self.counter    = 0
            self.best_epoch = self._epoch
            # Store on CPU to avoid holding a second full model in GPU memory
            self.best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict({k: v.to(device) for k, v in self.best_state.items()})


def make_head():
    return nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(512, NUM_CLASSES)
    ).to(device)


class CLIPClassifier(nn.Module):
    def __init__(self, clip_mdl, freeze_backbone=True):
        super().__init__()
        self.clip = clip_mdl
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, NUM_CLASSES))
        if freeze_backbone:
            for p in self.clip.parameters(): p.requires_grad = False

    def forward(self, x):
        return self.head(self.clip.encode_image(x).float())


def make_full_clf(head_state, unfreeze_blocks):
    clf = CLIPClassifier(clip_model, freeze_backbone=True).to(device)
    clf.head.load_state_dict(head_state)
    for block_idx in unfreeze_blocks:
        for p in clf.clip.visual.transformer.resblocks[block_idx].parameters():
            p.requires_grad = True
    if unfreeze_blocks:
        for p in clf.clip.visual.ln_post.parameters(): p.requires_grad = True
        clf.clip.visual.proj.requires_grad = True
    return clf


def train_head_epoch(head, loader, optimizer):
    head.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = head(embs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * embs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += embs.size(0)
    return total_loss / n, total_correct / n


@torch.no_grad()
def eval_head(head, loader):
    head.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        out  = head(embs)
        loss = criterion(out, labels)
        total_loss    += loss.item() * embs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += embs.size(0)
    return total_loss / n, total_correct / n


def train_one_epoch(clf, loader, optimizer, scheduler=None):
    clf.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = clf(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += imgs.size(0)
    if scheduler: scheduler.step()
    return total_loss / n, total_correct / n


@torch.no_grad()
def evaluate(clf, loader):
    clf.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = clf(imgs)
        loss = criterion(out, labels)
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += imgs.size(0)
    return total_loss / n, total_correct / n


print('Helper functions defined.')

In [ ]:
# ── CV runner ────────────────────────────────────────────────
def run_cv_config(config):
    config_name     = config['name']
    unfreeze_blocks = config['unfreeze_blocks']
    lr_backbone     = config['lr_backbone']
    fold_best_vals, fold_best_epochs, fold_a_epochs = [], [], []

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f'\n--- Config: {config_name} | Fold {fold_idx+1}/{K_FOLDS} ---')
        set_seed(SEED + fold_idx)

        # ── Phase A: head on cached embeddings ──
        tr_labs = torch.tensor(all_labels[tr_idx], dtype=torch.long)
        vl_labs = torch.tensor(all_labels[vl_idx], dtype=torch.long)
        tr_emb_ds = EmbeddingDataset(all_embeddings[tr_idx], tr_labs)
        vl_emb_ds = EmbeddingDataset(all_embeddings[vl_idx], vl_labs)
        tr_emb_loader = DataLoader(tr_emb_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        vl_emb_loader = DataLoader(vl_emb_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        head  = make_head()
        opt_a = optim.AdamW(head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
        es_a  = EarlyStopping(patience=PATIENCE_A)

        print(f'  Phase A (cached, max {MAX_EPOCHS_A} epochs, patience {PATIENCE_A}):')
        for epoch in range(MAX_EPOCHS_A):
            tr_loss, tr_acc = train_head_epoch(head, tr_emb_loader, opt_a)
            vl_loss, vl_acc = eval_head(head, vl_emb_loader)
            stop = es_a.step(vl_acc, head)
            print(f'    [{epoch+1:02d}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}')
            if stop:
                print(f'    Early stop. Best val {es_a.best_val:.4f} @ epoch {es_a.best_epoch}')
                break
        es_a.restore(head)
        fold_a_epochs.append(es_a.best_epoch)  # track Phase A epoch for budget

        if not unfreeze_blocks:
            fold_best_vals.append(es_a.best_val)
            fold_best_epochs.append(es_a.best_epoch)
            continue

        # ── Phase B: fine-tune on raw images ──
        # Reset backbone to original CLIP weights (keep folds independent)
        clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})

        train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), preprocess]) if USE_AUG else preprocess
        tr_samples = list(zip(all_paths[tr_idx].tolist(), all_labels[tr_idx].tolist()))
        vl_samples = list(zip(all_paths[vl_idx].tolist(), all_labels[vl_idx].tolist()))
        tr_ds = LabeledDataset(tr_samples, transform=train_tf)
        vl_ds = LabeledDataset(vl_samples, transform=preprocess)
        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
        vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        clf   = make_full_clf(head.state_dict(), unfreeze_blocks)
        opt_b = optim.AdamW([
            {'params': [p for p in clf.clip.parameters() if p.requires_grad], 'lr': lr_backbone},
            {'params': clf.head.parameters(), 'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        sch_b = optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=MAX_EPOCHS_B)
        es_b  = EarlyStopping(patience=PATIENCE_B)

        print(f'  Phase B ({len(unfreeze_blocks)} blocks @ lr={lr_backbone}, max {MAX_EPOCHS_B} epochs, patience {PATIENCE_B}):')
        for epoch in range(MAX_EPOCHS_B):
            tr_loss, tr_acc = train_one_epoch(clf, tr_loader, opt_b, sch_b)
            vl_loss, vl_acc = evaluate(clf, vl_loader)
            stop = es_b.step(vl_acc, clf)
            print(f'    [{epoch+1:02d}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}')
            if stop:
                print(f'    Early stop. Best val {es_b.best_val:.4f} @ epoch {es_b.best_epoch}')
                break
        es_b.restore(clf)

        torch.save({'model_state_dict': clf.state_dict(), 'val_acc': es_b.best_val},
                   os.path.join(CKPT_DIR, f'{config_name}_fold{fold_idx}_best.pt'))

        fold_best_vals.append(es_b.best_val)
        fold_best_epochs.append(es_b.best_epoch)

        # Reset backbone for next fold
        clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})

    return {
        'name':           config_name,
        'unfreeze':       len(unfreeze_blocks),
        'lr_backbone':    lr_backbone,
        'fold_vals':      fold_best_vals,
        'fold_epochs':    fold_best_epochs,
        'fold_a_epochs':  fold_a_epochs,          # Phase A best epochs per fold
        'cv_mean':        float(np.mean(fold_best_vals)),
        'cv_std':         float(np.std(fold_best_vals)),
        'median_epoch':   int(np.median(fold_best_epochs)),
        'median_a_epoch': int(np.median(fold_a_epochs)),  # Phase A budget for final retrain
    }


print('CV runner defined.')

In [ ]:
# ── Run CV for all configs ───────────────────────────────────
all_results = []
for config in CONFIGS:
    print(f'\n{"="*60}')
    print(f'Running CV: {config["name"]}')
    print(f'{"="*60}')
    result = run_cv_config(config)
    all_results.append(result)
    print(f'\n{config["name"]} → mean {result["cv_mean"]:.4f} ± {result["cv_std"]:.4f} | median epoch {result["median_epoch"]}')

In [ ]:
# ── Results table + 1-std-error model selection ──────────────
import math

print('\n── CV Results ──────────────────────────────────────────────────')
print(f'{"Config":<20} {"lr_bb":>7} {"mean":>7} {"std":>7} {"se":>7} {"med_B":>7} {"med_A":>7} {"folds"}')
print('-' * 85)
for r in all_results:
    se        = r['cv_std'] / math.sqrt(K_FOLDS)
    folds_str = ' '.join(f'{v:.4f}' for v in r['fold_vals'])
    print(f'{r["name"]:<20} {r["lr_backbone"]:>7.0e} {r["cv_mean"]:>7.4f} {r["cv_std"]:>7.4f} {se:>7.4f} {r["median_epoch"]:>7} {r["median_a_epoch"]:>7} {folds_str}')

# 1-std-error rule (strict: std / sqrt(K))
best_mean = max(r['cv_mean'] for r in all_results)
best_se   = next(r['cv_std'] / math.sqrt(K_FOLDS) for r in all_results if r['cv_mean'] == best_mean)
threshold = best_mean - best_se

candidates = [r for r in all_results if r['cv_mean'] >= threshold]
selected   = min(candidates, key=lambda r: r['unfreeze'])

print(f'\n── Model Selection (1-SE rule, strict: std/√{K_FOLDS}) ────────────')
print(f'Best mean:  {best_mean:.4f}')
print(f'SE:         {best_se:.4f}')
print(f'Threshold:  {threshold:.4f}')
print(f'Candidates: {[r["name"] for r in candidates]}')
print(f'Selected:   {selected["name"]} (simplest within 1 SE of best)')
print(f'Phase A budget: {selected["median_a_epoch"]} epochs')
print(f'Phase B budget: {selected["median_epoch"]} epochs')

In [ ]:
# ── Final retrain on ALL data ────────────────────────────────
# Phase A budget = median_a_epoch from CV (not the full 60 ceiling)
# Phase B budget = median_epoch from CV

FINAL_CONFIG    = selected
FINAL_EPOCHS_A  = FINAL_CONFIG['median_a_epoch']
FINAL_EPOCHS_B  = FINAL_CONFIG['median_epoch'] if FINAL_CONFIG['unfreeze'] else 0
FINAL_LR_BB     = FINAL_CONFIG['lr_backbone']

print(f'Final retrain: {FINAL_CONFIG["name"]}')
print(f'Phase A: {FINAL_EPOCHS_A} epochs (from CV median)')
print(f'Phase B: {FINAL_EPOCHS_B} epochs (from CV median) @ lr_backbone={FINAL_LR_BB}')

# Reset backbone
clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})
set_seed(SEED)

all_labs_tensor = torch.tensor(all_labels, dtype=torch.long)
all_emb_ds      = EmbeddingDataset(all_embeddings, all_labs_tensor)
all_emb_loader  = DataLoader(all_emb_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

final_head = make_head()
opt_fa     = optim.AdamW(final_head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

print(f'\nPhase A — all data, {FINAL_EPOCHS_A} epochs:')
for epoch in range(FINAL_EPOCHS_A):
    tr_loss, tr_acc = train_head_epoch(final_head, all_emb_loader, opt_fa)
    print(f'  [{epoch+1:02d}] train {tr_acc:.4f}')

if FINAL_CONFIG['unfreeze']:
    train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), preprocess]) if USE_AUG else preprocess
    all_samples    = list(zip(all_paths.tolist(), all_labels.tolist()))
    all_raw_ds     = LabeledDataset(all_samples, transform=train_tf)
    all_raw_loader = DataLoader(all_raw_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    final_clf = make_full_clf(final_head.state_dict(), FINAL_CONFIG['unfreeze_blocks'])
    opt_fb    = optim.AdamW([
        {'params': [p for p in final_clf.clip.parameters() if p.requires_grad], 'lr': FINAL_LR_BB},
        {'params': final_clf.head.parameters(), 'lr': LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    sch_fb = optim.lr_scheduler.CosineAnnealingLR(opt_fb, T_max=FINAL_EPOCHS_B)

    print(f'\nPhase B — all data, {FINAL_EPOCHS_B} epochs:')
    for epoch in range(FINAL_EPOCHS_B):
        tr_loss, tr_acc = train_one_epoch(final_clf, all_raw_loader, opt_fb, sch_fb)
        print(f'  [{epoch+1:02d}] train {tr_acc:.4f}')
else:
    final_clf = CLIPClassifier(clip_model, freeze_backbone=True).to(device)
    final_clf.head.load_state_dict(final_head.state_dict())

torch.save({'model_state_dict': final_clf.state_dict(), 'config': FINAL_CONFIG['name']},
           os.path.join(CKPT_DIR, 'final_model.pt'))
print('\nSaved: checkpoints_kfold/final_model.pt')

In [ ]:
# ── Submission ───────────────────────────────────────────────
test_ds     = TestDataset(TEST_DIR, transform=preprocess)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Load final model (run this cell standalone if needed)
ckpt = torch.load(os.path.join(CKPT_DIR, 'final_model.pt'), map_location=device)
for p in final_clf.parameters(): p.requires_grad = True
final_clf.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded final model: {ckpt["config"]}')

final_clf.eval()
ids, preds = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader):
        imgs = imgs.to(device)
        out  = final_clf(imgs)
        ids.extend(names)
        preds.extend(out.argmax(1).cpu().tolist())

sub = pd.DataFrame({'ID': ids, 'Label': preds})
sub.to_csv('submission_kfold.csv', index=False)
print(sub.head(10))
print(f'Saved submission_kfold.csv ({len(sub)} rows)')